# 1.0 Dual Engine Analytics System Overview

- This project implements a **Dual Engine Analytics System (SQL + Python)** to analyze restaurant performance data.
- The system integrates **order-level data (primary fact table)** and **order item-level data (line-item detail)** to enable full revenue and operational analysis.
- **SQL (DuckDB)** is used for:
  - Data ingestion
  - Cleaning and standardization
  - Joins and aggregations
  - KPI and business query development
- **Python (pandas + matplotlib)** is used for:
  - Validation of SQL outputs
  - Time-based analysis
  - Visualization
  - Business insight generation
- The workflow follows a structured pipeline:
  - Data ingestion → Cleaning → Validation → Analysis → Business recommendations
- The objective is to build a **portfolio-grade analytics system** that demonstrates:
  - Strong SQL fundamentals
  - Clean data engineering practices
  - Business-focused analytical thinking
- All outputs are designed to connect data insights to **revenue growth, operational efficiency, and decision-making impact**.

## 1.1 Data Ingestion (DuckDB Setup)

- Initialize a DuckDB connection to serve as the in-notebook SQL engine.
- Load all raw order-level datasets into SQL tables.
- Ingest quarterly files across 2024–2025.
- Maintain separation of datasets before transformation and union.
- Validate successful ingestion of all tables.

In [27]:
!pip install duckdb

In [28]:
import duckdb

# Create DuckDB connection
con = duckdb.connect(database=':memory:')

# -------------------------
# LOAD ORDER-LEVEL DATA
# -------------------------

con.execute("""
CREATE TABLE orders_2025_q1 AS
SELECT * FROM read_csv_auto('data/raw/orders_raw/2025/orders_2025_q1.csv');
""")

con.execute("""
CREATE TABLE orders_2025_q2 AS
SELECT * FROM read_csv_auto('data/raw/orders_raw/2025/orders_2025_q2.csv');
""")

con.execute("""
CREATE TABLE orders_2025_q3 AS
SELECT * FROM read_csv_auto('data/raw/orders_raw/2025/orders_2025_q3.csv');
""")

con.execute("""
CREATE TABLE orders_2024_q3 AS
SELECT * FROM read_csv_auto('data/raw/orders_raw/2024/orders_2024_q3.csv');
""")

con.execute("""
CREATE TABLE orders_2024_q4 AS
SELECT * FROM read_csv_auto('data/raw/orders_raw/2024/orders_2024_q4.csv');
""")

# -------------------------
# VALIDATION
# -------------------------

con.execute("SHOW TABLES").fetchall()

[('orders_2024_q3',),
 ('orders_2024_q4',),
 ('orders_2025_q1',),
 ('orders_2025_q2',),
 ('orders_2025_q3',)]

## 1.2 Data Inspection (SQL)

- Inspect the structure of each raw order table to understand column names, data types, and formatting.
- Verify that all quarterly datasets share a consistent schema before cleaning and union.
- Identify potential issues such as:
  - timestamps stored as text
  - monetary fields stored as strings
  - null or missing values
  - inconsistent column naming
- Preview sample rows to understand the raw data layout before transformation.
- This step informs the cleaning strategy used in the next section.

In [29]:
# Inspect schema for each quarterly table
con.execute("DESCRIBE orders_2025_q1").fetchdf()
con.execute("DESCRIBE orders_2025_q2").fetchdf()
con.execute("DESCRIBE orders_2025_q3").fetchdf()
con.execute("DESCRIBE orders_2024_q3").fetchdf()
con.execute("DESCRIBE orders_2024_q4").fetchdf()

# Preview sample rows
con.execute("SELECT * FROM orders_2025_q1 LIMIT 5").fetchdf()
con.execute("SELECT * FROM orders_2024_q3 LIMIT 5").fetchdf()

,Order #,Opened,# of Guests,Server,Table,Discount Amount,Amount,Tax,Tip,Gratuity
0,17,9/4/24 10:54 AM,1,Giovanni Cortes,None,0.0,8.74,0.87,0.0,0.0
1,18,9/4/24 10:59 AM,1,Jessica Mccullar,None,0.0,14.00,1.40,0.0,0.0
2,19,9/4/24 10:59 AM,3,Zachary O'Brian,B3,0.0,31.25,3.12,0.0,0.0
3,20,9/4/24 11:04 AM,1,Jessica Mccullar,None,0.0,5.25,0.53,0.0,0.0
4,22,9/4/24 11:10 AM,1,Zachary O'Brian,10,0.0,14.24,1.42,0.0,0.0


## 1.3 Data Cleaning Strategy

- Standardize all column names to lowercase with underscores for SQL compatibility.
- Rename key fields:
  - "Order #" → order_id
  - "Opened" → order_datetime
  - "# of Guests" → guest_count
  - "Amount" → order_total
- Convert the "Opened" column from string to proper timestamp format.
- Prepare timestamp for timezone handling (Central Time origin).
- Ensure all monetary fields are stored as numeric types.
- Validate no null values exist in critical fields such as order_id and order_total.
- Confirm schema consistency across all quarterly tables before union.
- This step prepares the dataset for reliable aggregation and time-based analysis.

In [30]:
con.execute("""
CREATE OR REPLACE TABLE orders_clean AS

WITH orders_2025_q1_clean AS (
    SELECT
        CAST("Order #" AS BIGINT) AS order_id,
        STRPTIME("Opened", '%m/%d/%y %I:%M %p') AS order_datetime,
        CAST("# of Guests" AS INTEGER) AS guest_count,
        "Server" AS server_name,
        "Table" AS table_name,
        CAST("Discount Amount" AS DOUBLE) AS discount_amount,
        CAST("Amount" AS DOUBLE) AS order_total,
        CAST("Tax" AS DOUBLE) AS tax_amount,
        CAST("Tip" AS DOUBLE) AS tip_amount,
        CAST("Gratuity" AS DOUBLE) AS gratuity_amount,
        2025 AS year,
        'Q1' AS quarter
    FROM orders_2025_q1
),

orders_2025_q2_clean AS (
    SELECT
        CAST("Order #" AS BIGINT) AS order_id,
        STRPTIME("Opened", '%m/%d/%y %I:%M %p') AS order_datetime,
        CAST("# of Guests" AS INTEGER) AS guest_count,
        "Server" AS server_name,
        "Table" AS table_name,
        CAST("Discount Amount" AS DOUBLE) AS discount_amount,
        CAST("Amount" AS DOUBLE) AS order_total,
        CAST("Tax" AS DOUBLE) AS tax_amount,
        CAST("Tip" AS DOUBLE) AS tip_amount,
        CAST("Gratuity" AS DOUBLE) AS gratuity_amount,
        2025 AS year,
        'Q2' AS quarter
    FROM orders_2025_q2
),

orders_2025_q3_clean AS (
    SELECT
        CAST("Order #" AS BIGINT) AS order_id,
        STRPTIME("Opened", '%m/%d/%y %I:%M %p') AS order_datetime,
        CAST("# of Guests" AS INTEGER) AS guest_count,
        "Server" AS server_name,
        "Table" AS table_name,
        CAST("Discount Amount" AS DOUBLE) AS discount_amount,
        CAST("Amount" AS DOUBLE) AS order_total,
        CAST("Tax" AS DOUBLE) AS tax_amount,
        CAST("Tip" AS DOUBLE) AS tip_amount,
        CAST("Gratuity" AS DOUBLE) AS gratuity_amount,
        2025 AS year,
        'Q3' AS quarter
    FROM orders_2025_q3
),

orders_2024_q3_clean AS (
    SELECT
        CAST("Order #" AS BIGINT) AS order_id,
        STRPTIME("Opened", '%m/%d/%y %I:%M %p') AS order_datetime,
        CAST("# of Guests" AS INTEGER) AS guest_count,
        "Server" AS server_name,
        "Table" AS table_name,
        CAST("Discount Amount" AS DOUBLE) AS discount_amount,
        CAST("Amount" AS DOUBLE) AS order_total,
        CAST("Tax" AS DOUBLE) AS tax_amount,
        CAST("Tip" AS DOUBLE) AS tip_amount,
        CAST("Gratuity" AS DOUBLE) AS gratuity_amount,
        2024 AS year,
        'Q3' AS quarter
    FROM orders_2024_q3
),

orders_2024_q4_clean AS (
    SELECT
        CAST("Order #" AS BIGINT) AS order_id,
        STRPTIME("Opened", '%m/%d/%y %I:%M %p') AS order_datetime,
        CAST("# of Guests" AS INTEGER) AS guest_count,
        "Server" AS server_name,
        "Table" AS table_name,
        CAST("Discount Amount" AS DOUBLE) AS discount_amount,
        CAST("Amount" AS DOUBLE) AS order_total,
        CAST("Tax" AS DOUBLE) AS tax_amount,
        CAST("Tip" AS DOUBLE) AS tip_amount,
        CAST("Gratuity" AS DOUBLE) AS gratuity_amount,
        2024 AS year,
        'Q4' AS quarter
    FROM orders_2024_q4
)

SELECT * FROM orders_2025_q1_clean
UNION ALL
SELECT * FROM orders_2025_q2_clean
UNION ALL
SELECT * FROM orders_2025_q3_clean
UNION ALL
SELECT * FROM orders_2024_q3_clean
UNION ALL
SELECT * FROM orders_2024_q4_clean
""")

con.execute("SELECT * FROM orders_clean LIMIT 10").fetchdf()

,order_id,order_datetime,guest_count,server_name,table_name,discount_amount,order_total,tax_amount,tip_amount,gratuity_amount,year,quarter
0,1,2025-01-02 11:02:00,1,Jessica Mccullar,None,0.0,9.99,1.00,2.0,0.0,2025,Q1
1,2,2025-01-02 11:03:00,1,Jessica Mccullar,None,0.0,11.49,1.15,0.0,0.0,2025,Q1
2,6001,2025-01-02 11:10:00,1,Zachary O'Brian,4,0.0,26.99,2.69,0.0,0.0,2025,Q1
3,3,2025-01-02 11:24:00,1,Zachary O'Brian,1,0.0,36.23,3.62,20.0,0.0,2025,Q1
4,4,2025-01-02 11:53:00,1,Jessica Mccullar,None,0.0,39.24,3.92,0.0,0.0,2025,Q1
5,5,2025-01-02 11:54:00,1,Jessica Mccullar,None,0.0,12.99,1.29,0.0,0.0,2025,Q1
6,6,2025-01-02 12:01:00,1,Zachary O'Brian,2,0.0,49.20,4.92,0.0,0.0,2025,Q1
7,7,2025-01-02 12:03:00,1,Zachary O'Brian,4,0.0,27.50,2.75,0.0,0.0,2025,Q1
8,8,2025-01-02 12:05:00,1,Zachary O'Brian,11,0.0,27.97,2.80,0.0,0.0,2025,Q1
9,9,2025-01-02 12:22:00,1,Zachary O'Brian,None,0.0,7.75,0.78,0.0,0.0,2025,Q1


## 1.5 Data Validation

- Validate that all quarterly datasets were successfully combined into the `orders_clean` table.
- Compare row counts between raw tables and the cleaned table to ensure no data loss.
- Check for null values in critical fields such as `order_id`, `order_datetime`, and `order_total`.
- Confirm that timestamp conversion was successful and no parsing errors occurred.
- Verify that `year` and `quarter` fields are correctly assigned.
- This step ensures the dataset is reliable before proceeding to analysis.

In [31]:
# -------------------------
# ROW COUNT VALIDATION
# -------------------------

# Count rows in raw tables
raw_counts = con.execute("""
SELECT 
    (SELECT COUNT(*) FROM orders_2025_q1) +
    (SELECT COUNT(*) FROM orders_2025_q2) +
    (SELECT COUNT(*) FROM orders_2025_q3) +
    (SELECT COUNT(*) FROM orders_2024_q3) +
    (SELECT COUNT(*) FROM orders_2024_q4) AS total_raw_rows
""").fetchdf()

# Count rows in cleaned table
clean_count = con.execute("""
SELECT COUNT(*) AS total_clean_rows FROM orders_clean
""").fetchdf()

raw_counts, clean_count


# -------------------------
# NULL CHECKS
# -------------------------

con.execute("""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_id,
    SUM(CASE WHEN order_datetime IS NULL THEN 1 ELSE 0 END) AS null_order_datetime,
    SUM(CASE WHEN order_total IS NULL THEN 1 ELSE 0 END) AS null_order_total
FROM orders_clean
""").fetchdf()


# -------------------------
# TIMESTAMP VALIDATION
# -------------------------

con.execute("""
SELECT 
    MIN(order_datetime) AS min_datetime,
    MAX(order_datetime) AS max_datetime
FROM orders_clean
""").fetchdf()


# -------------------------
# YEAR / QUARTER VALIDATION
# -------------------------

con.execute("""
SELECT 
    year,
    quarter,
    COUNT(*) AS row_count
FROM orders_clean
GROUP BY year, quarter
ORDER BY year, quarter
""").fetchdf()

,year,quarter,row_count
0,2024,Q3,1761
1,2024,Q4,5510
2,2025,Q1,5654
3,2025,Q2,6234
4,2025,Q3,2997


## 1.4 Orders Data Cleaning

- Clean and standardize all quarterly order tables.
- Rename columns to SQL-friendly format (lowercase, underscores).
- Convert the "Opened" field into a proper timestamp column.
- Ensure all monetary fields are numeric.
- Add `year` and `quarter` fields for time-based analysis.
- Combine all quarterly datasets into a unified `orders_clean` table using UNION ALL.
- This creates the primary fact table for downstream analysis.

## 1.5 Data Validation

- Validate that all quarterly datasets were successfully combined into `orders_clean`.
- Compare row counts between raw and cleaned tables to ensure no data loss.
- Check for null values in key fields such as:
  - order_id
  - order_datetime
  - order_total
- Confirm timestamp parsing was successful.
- Verify correct assignment of `year` and `quarter`.
- This ensures the dataset is reliable before analysis.

In [32]:
con.execute("""
COPY orders_clean TO 'data/cleaned/orders_clean.csv' (HEADER, DELIMITER ',');
""")

## 1.6 SQL Validation Layer

- Use SQL directly within DuckDB to validate the cleaned dataset.
- Demonstrates ability to perform data validation and aggregation using SQL.
- Confirms row-level integrity of the cleaned dataset.

In [33]:
# SQL validation directly in DuckDB

con.execute("""
SELECT
    COUNT(*) AS total_orders,
    MIN(order_datetime) AS min_date,
    MAX(order_datetime) AS max_date
FROM orders_clean
""").fetchdf()

,total_orders,min_date,max_date
0,22156,2024-09-04 10:54:00,2025-08-18 11:13:00


In [34]:
con.execute("""
SELECT
    year,
    quarter,
    COUNT(*) AS orders
FROM orders_clean
GROUP BY year, quarter
ORDER BY year, quarter
""").fetchdf()

,year,quarter,orders
0,2024,Q3,1761
1,2024,Q4,5510
2,2025,Q1,5654
3,2025,Q2,6234
4,2025,Q3,2997


## 1.7 Export Clean Dataset

- Export the fully cleaned, row-level dataset from DuckDB.
- Enforce correct data types (e.g., datetime) upstream.
- Establish a single source of truth for all downstream notebooks.

In [41]:
import os
import pandas as pd

# ==============================
# Step 1: Confirm correct table exists
# ==============================

con.execute("SHOW TABLES").fetchdf()

# ==============================
# Step 2: Extract FULL row-level dataset
# ==============================

orders_clean_df = con.execute("""
SELECT * FROM orders_clean
""").fetchdf()

# ==============================
# Step 3: FIX DATA TYPES (UPSTREAM 🔥)
# ==============================

orders_clean_df["order_datetime"] = pd.to_datetime(orders_clean_df["order_datetime"])

# ==============================
# Step 4: Create folder if needed
# ==============================

os.makedirs("../data/cleaned", exist_ok=True)

# ==============================
# Step 5: Export dataset
# ==============================

orders_clean_df.to_csv("../data/cleaned/orders_clean.csv", index=False)

# ==============================
# Step 6: Confirm export
# ==============================

print("orders_clean.csv exported successfully")
print("Rows exported:", len(orders_clean_df))
print("Columns:", list(orders_clean_df.columns))

orders_clean.csv exported successfully
Rows exported: 22156
Columns: ['order_id', 'order_datetime', 'guest_count', 'server_name', 'table_name', 'discount_amount', 'order_total', 'tax_amount', 'tip_amount', 'gratuity_amount', 'year', 'quarter']
